# Revisión 8 en Colab (sólo procesador)

Corre en Colab el cómputo de la revisión 8 del protocolo `anomalias_red_v1`. Es el mismo
procedimiento que `docs/runbook_pc_revision8.md`, pero sin otra instancia de Claude: este cuaderno
sólo **calcula**, y el análisis y el registro los hace Claude en la lap con los resultados
descargados.

**Antes de empezar:**
1. **Entorno de ejecución → Cambiar tipo de entorno → CPU** (con RAM alta si tu plan la tiene).
   **No uses GPU:** cambia los números y no da exactamente lo mismo dos veces.
2. **Sube a tu Google Drive, en `Mi unidad/tesis_rama/raw/`,** los 7 zips de la lap:
   `data/raw/20RAMA.zip` … `26RAMA.zip`. Unos 38 MB, una sola vez.
3. **Deja esta pestaña abierta mientras corre:** en Colab Pro, cerrarla puede cortar la sesión.

**Reglas:**
- No cambies nada del diseño.
- **Si una comprobación dice DISTINTO o falla, detente** y avísale a Claude con lo que salió.
- Las celdas largas corren en segundo plano y guardan su registro en Drive.

## 1. Drive y repositorio

In [ ]:
REPO = 'https://github.com/JoseLili/lorawan11-rama-detection.git'
DRIVE = '/content/drive/MyDrive/tesis_rama'          # raw/ (zips), registros/ y resultados/
RAIZ = '/content/lorawan11-rama-detection'
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, json, time
for d in ('raw', 'registros', 'resultados'):
    os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
if os.path.isdir(RAIZ):
    print(subprocess.run(['git', '-C', RAIZ, 'pull'], capture_output=True, text=True).stdout)
else:
    subprocess.run(['git', 'clone', REPO, RAIZ], check=True)
os.chdir(RAIZ)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

## 2. Datos crudos: copiar y verificar sus SHA-256 (todos deben decir «OK»)

In [ ]:
!mkdir -p data/raw && cp {DRIVE}/raw/*RAMA.zip data/raw/ && ls -la data/raw
!grep -E '  [0-9]{2}RAMA\.zip$' docs/checksums_2020_2026.txt | (cd data/raw && sha256sum -c -)
# Si no subiste los zips a Drive: from google.colab import files; files.upload()  (y muévelos a data/raw/)

## 3. Entorno con las versiones exactas de la lap (~5 minutos)

Python 3.14.7 y `requirements-lock.txt`. Si alguna versión no se instala, **detente y avisa**; no
uses otras.

In [ ]:
!pip -q install uv
!uv python install 3.14.7
!uv venv --python 3.14.7 .venv
!uv pip install --python .venv/bin/python -r requirements-lock.txt 2>&1 | tail -3
v = subprocess.run(['.venv/bin/python', '-c', 'import sys, tensorflow as tf, keras, numpy, pandas, sklearn; '
                    'print(sys.version.split()[0], tf.__version__, keras.__version__, numpy.__version__, '
                    'pandas.__version__, sklearn.__version__)'], capture_output=True, text=True).stdout.split()
esperado = ['3.14.7', '2.22.0-rc0', '3.16.0.dev2026092204', '2.5.3', '3.0.6', '1.9.1']
print(v)
assert v == esperado, f'versiones distintas a las de la lap: DETENTE y avisa ({v})'
!nproc; lscpu | grep 'Model name'; free -g | sed -n 2p

## 4. Regenerar planes y matrices, y correr las pruebas

`git status` no debe mostrar cambios: los planes y el resumen de matrices tienen que salir
idénticos a los del repo. Deben pasar las 265 pruebas.

In [ ]:
!.venv/bin/python experiments/anomalias_red/generar_planes.py 2>&1 | tail -3
!.venv/bin/python experiments/anomalias_red/generar_matrices.py 2>&1 | tail -3
cambios = subprocess.run(['git', 'status', '--short', 'results/anomalias_red_v1/'], capture_output=True, text=True).stdout
print(cambios or '(sin cambios: los datos se reproducen)')
assert not cambios.strip(), 'Colab no reproduce los planes o las matrices: DETENTE y avisa'
!.venv/bin/python -m pytest -q 2>&1 | tail -2

## 5. Prueba de equivalencia (~8 minutos en la lap)

Repite la configuración 3 de la curva de aprendizaje con δ = 100 % y la compara época por época
con la lap. **Si sale DISTINTO, detente:** el tesista decide con Claude si se sigue en Colab.

In [ ]:
t0 = time.time()
!.venv/bin/python experiments/anomalias_red/diagnostico_curva_aprendizaje_2022.py --delta 100 --solo 3 --salida /content/equivalencia > {DRIVE}/registros/equivalencia.log 2>&1
print(f'tardó {(time.time() - t0) / 60:.1f} min')
import pandas as pd
a = pd.read_csv('results/anomalias_red_v1/curva_aprendizaje_2022/resultados.csv')
a = a[(a.delta_pct == 100) & (a.configuracion == 3)].reset_index(drop=True)
b = pd.read_csv('/content/equivalencia/resultados.csv')
c = ['epoca', 'deteccion_2022_fa5_pct', 'auc_2022', 'val_loss']
print(pd.concat([a[c], b[c].add_suffix('_colab')], axis=1).to_string())
print('IDÉNTICO' if a[c].equals(b[c]) else 'DISTINTO: detente y avisa')

## 6. Elección de la forma con 2022, dos veces (en segundo plano)

Son 5 modelos × δ = 20 y 100 % × 10 épocas, dos veces. En la lap serían ~2.7 h. Corre en segundo
plano: la celda siguiente muestra el avance, y puedes correrla las veces que quieras.

In [ ]:
orden = (f'.venv/bin/python experiments/anomalias_red/diagnostico_vecinas_2022.py > {DRIVE}/registros/vecinas_1.log 2>&1 && '
         f'.venv/bin/python experiments/anomalias_red/diagnostico_vecinas_2022.py --salida /content/vecinas_c2 '
         f'> {DRIVE}/registros/vecinas_2.log 2>&1 && echo FIN >> {DRIVE}/registros/vecinas_2.log')
subprocess.Popen(['nohup', 'bash', '-c', orden], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
print('lanzado', time.strftime('%H:%M'))

In [ ]:
# Avance: 50 épocas por corrida (5 modelos × 2 δ × 10 épocas).
for k in (1, 2):
    p = f'{DRIVE}/registros/vecinas_{k}.log'
    if os.path.exists(p):
        txt = open(p).read()
        n = txt.count("'epoca'")
        print(f'corrida {k}: {n} de 50 épocas', '· FIN' if 'FIN' in txt else '',
              '· ERROR, revisa el registro' if 'Traceback' in txt else '')

## 7. Comparar las dos corridas y guardar el resultado

In [ ]:
for f in ('resultados.csv', 'resumen.csv', 'eleccion.json'):
    igual = open(f'results/anomalias_red_v1/vecinas_2022/{f}', 'rb').read() == open(f'/content/vecinas_c2/{f}', 'rb').read()
    print(f, 'idéntico' if igual else 'DISTINTO: detente y avisa')
print(pd.read_csv('results/anomalias_red_v1/vecinas_2022/resumen.csv').round(3).to_string(index=False))
print({k: v for k, v in json.load(open('results/anomalias_red_v1/vecinas_2022/eleccion.json')).items() if k != 'vecinas'})
!cd {RAIZ} && zip -qr {DRIVE}/resultados/revision8_vecinas_2022.zip results/anomalias_red_v1/vecinas_2022 && ls -la {DRIVE}/resultados/
from google.colab import files
files.download(f'{DRIVE}/resultados/revision8_vecinas_2022.zip')

## ALTO: lleva el resultado a Claude

Descarga `revision8_vecinas_2022.zip`, que también queda en Drive, en `tesis_rama/resultados/`. En
la lap, Claude lo revisa, lo registra en `progreso.md` y en el protocolo, y te explica qué forma
ganó. **No sigas con la corrida completa sin su visto bueno.** La sesión de Colab se puede cerrar:
el paso 8 vuelve a preparar todo.

## 8. Corrida completa (sólo con el visto bueno)

Si se reinició la sesión, corre antes las celdas 1 a 4. Pon la forma elegida y `APROBADO = True`.
Por cada δ corre el LSTM dos veces, el Random Forest y la regla dos veces, y compara todo. En la
lap serían ~2 h.

In [ ]:
FORMA = 'a14'          # la forma elegida con 2022 (eleccion.json)
APROBADO = False       # True sólo con el visto bueno del tesista
assert APROBADO, 'falta el visto bueno'
orden = f'bash experiments/anomalias_red/correr_revision8.sh {FORMA} > {DRIVE}/registros/revision8_total.log 2>&1'
subprocess.Popen(['nohup', 'bash', '-c', orden], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
print('lanzado', time.strftime('%H:%M'))

In [ ]:
# Avance de la corrida completa: las comparaciones deben decir «idéntico» y terminar con FIN.
print(open(f'{DRIVE}/registros/revision8_total.log').read())

In [ ]:
!cd {RAIZ} && zip -qr {DRIVE}/resultados/revision8_completa.zip results/anomalias_red_v1/lstm_sup_vecinas_* results/anomalias_red_v1/rf_vecinas_* results/anomalias_red_v1/regla_vecinas_* logs_revision8 -x '*.joblib' && ls -la {DRIVE}/resultados/
from google.colab import files
files.download(f'{DRIVE}/resultados/revision8_completa.zip')